# Undo a pixel shift or rotation in the inverted latent

Five images, the same initial Gaussian seeds as samples 0–4 of the 100-image registration run. Each image is shifted and rotated in pixel space, DDIM-inverted, and saved. The notebook then applies the opposite shift or rotation on the inverted latent and compares both versions with the original Gaussian.

Run this from the repository root in a HiPerGator Jupyter session with the `gaussian-rotation` kernel and one GPU. Nothing here opens a desktop window. The run cell prints a step count and a time-left estimate after every generation and every inversion.

Outputs go to `results/geometric_registration/latent_undo_n5`, which is gitignored and does not touch `run_001`. A finished sample is skipped if you run the cell again.

## What each sample does

`z` is the Gaussian latent passed to the sampler. The image is produced with Stable Diffusion 1.5, DDIM, 50 steps, guidance 7.5, 512×512, and the prompt "a photograph of an astronaut riding a horse". That image is copied through the same conditions as the 100-image run: no transform, horizontal and vertical shifts of 1, 2, 5, and 10 pixels, and rotations of 1, 2, 5, and 10 degrees. New pixels are black. Shifts are nearest-neighbor, right or down. Rotations are bicubic and counterclockwise. `z_hat` is the fully noised end of a DDIM inversion of that edited image.

The score is still `pearsonr(|z|, |compared latent|)`.

## The latent undo

The VAE compresses each axis by 8, so an `n`-pixel image shift becomes an `n/8` latent-pixel shift. The undo moves the inverted latent the other way: left for a right image shift, up for a down image shift, clockwise for a counterclockwise image rotation. One and two pixels are fractional latent shifts (0.125 and 0.25), so those undos are nearest-neighbor and cannot be exact. Vacated latent sites are filled with 0, which is not the VAE code of a black pixel. This is a coordinate correction, not a claim that inversion is equivariant.

Five samples times 13 conditions is 5 generations and 65 inversions. On the earlier jobs that was a few minutes of GPU time after the weights loaded. The printed estimate replaces that guess once steps start finishing.


## 1. Configuration


In [ ]:
import importlib.util
import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from experiments.latent_geometry import undo_image_transform

_geom_spec = importlib.util.spec_from_file_location(
    "geometric_registration_experiment",
    ROOT / "scripts" / "run_geometric_registration_experiment.py",
)
geom = importlib.util.module_from_spec(_geom_spec)
_geom_spec.loader.exec_module(geom)

NUM_SAMPLES = 5
SEED = 42
# Draw the full 100-image seed list and keep the first five, so these Gaussians
# are samples 0-4 of results/geometric_registration/run_001.
REFERENCE_SAMPLE_COUNT = 100
OUTPUT_DIR = Path("results/geometric_registration/latent_undo_n5")
LATENT_DIR = OUTPUT_DIR / "latents"
EXAMPLE_DIR = OUTPUT_DIR / "examples"

MODEL_ID = "runwayml/stable-diffusion-v1-5"
NUM_INFERENCE_STEPS = 50
DEVICE = "cuda"
DTYPE = "float16"
IMAGE_SIZE = 512
PROMPT = "a photograph of an astronaut riding a horse"
GUIDANCE_SCALE = 7.5
VAE_SCALE_FACTOR = 8

CONDITIONS = geom.conditions()
FAMILIES = ["horizontal_translation", "vertical_translation", "rotation"]
FAMILY_TITLES = {
    "horizontal_translation": "horizontal shift (pixels right)",
    "vertical_translation": "vertical shift (pixels down)",
    "rotation": "rotation (degrees counterclockwise)",
}
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3


def fmt_duration(seconds: float) -> str:
    seconds = max(0, int(round(seconds)))
    minutes, secs = divmod(seconds, 60)
    hours, minutes = divmod(minutes, 60)
    if hours:
        return f"{hours}h {minutes}m {secs}s"
    if minutes:
        return f"{minutes}m {secs}s"
    return f"{secs}s"


def latent_key(kind: str, magnitude: float) -> str:
    return f"z_hat__{kind}__{magnitude:.1f}"


def sample_seeds() -> list[int]:
    master = np.random.default_rng(SEED)
    drawn = master.integers(0, np.iinfo(np.int32).max, size=REFERENCE_SAMPLE_COUNT)
    return [int(v) for v in drawn[:NUM_SAMPLES]]


def sample_path(sample_id: int) -> Path:
    return LATENT_DIR / f"sample_{sample_id:03d}.pt"


def sample_is_complete(sample_id: int) -> bool:
    path = sample_path(sample_id)
    if not path.is_file():
        return False
    blob = torch.load(path, map_location="cpu", weights_only=True)
    if "z" not in blob:
        return False
    return all(latent_key(kind, magnitude) in blob for kind, magnitude in CONDITIONS)


print(f"samples {NUM_SAMPLES}, conditions {len(CONDITIONS)}")
print(f"output {OUTPUT_DIR}")
print("seeds", sample_seeds())


## 2. Generate, invert, and save

The generation pipeline is loaded, the five images are produced, and that pipeline is released before the inversion pipeline is loaded. After every finished sample the latent file and `correlations.csv` are rewritten, so a stopped run continues from the next sample.

`correlation_received` compares `|z|` with `|z_hat|`. `correlation_undone` compares `|z|` with the absolute value of the geometrically corrected latent. Baseline undo is a copy, so those two scores match.


In [ ]:
class StepClock:
    def __init__(self, total: int, label: str):
        self.total = total
        self.label = label
        self.done = 0
        self.started = time.perf_counter()
        self.durations: list[float] = []
        self._tick = self.started

    def begin(self) -> None:
        self._tick = time.perf_counter()

    def end(self, detail: str) -> None:
        now = time.perf_counter()
        duration = now - self._tick
        self.durations.append(duration)
        self.done += 1
        average = sum(self.durations) / len(self.durations)
        remaining_steps = self.total - self.done
        eta = average * remaining_steps
        print(
            f"[{self.label} {self.done}/{self.total}] {detail} | "
            f"this step {duration:.1f}s, avg {average:.1f}s | "
            f"elapsed {fmt_duration(now - self.started)} | "
            f"about {fmt_duration(eta)} left ({remaining_steps} steps)",
            flush=True,
        )


def _load_saved(sample_id: int) -> dict[str, torch.Tensor]:
    return torch.load(sample_path(sample_id), map_location="cpu", weights_only=True)


def _scores_for_blob(blob: dict[str, torch.Tensor]) -> list[dict]:
    z = blob["z"].numpy()
    rows = []
    for kind, magnitude in CONDITIONS:
        z_hat = blob[latent_key(kind, magnitude)].numpy()
        undone = undo_image_transform(z_hat, kind, magnitude, VAE_SCALE_FACTOR)
        if z.shape != z_hat.shape or z.shape != undone.shape:
            raise RuntimeError(f"shape mismatch for {kind} {magnitude}: {z.shape} {z_hat.shape} {undone.shape}")
        received = geom.magnitude_correlation(z, z_hat)
        corrected = geom.magnitude_correlation(z, undone)
        if kind == "baseline" and abs(received - corrected) > 1e-5:
            raise RuntimeError("baseline undo changed the correlation")
        rows.append({
            "sample_id": None,
            "transformation_type": kind,
            "magnitude": magnitude,
            "correlation_received": received,
            "correlation_undone": corrected,
        })
    return rows


def run_experiment() -> pd.DataFrame:
    if DEVICE == "cuda" and not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is not available. Run this cell on a one-GPU HiPerGator session with the gaussian-rotation kernel."
        )
    from experiments.generate_and_invert import (
        _dtype,
        _git_commit,
        _load_generation_pipeline,
        _load_inversion_pipeline,
        _recovered_latent,
        _release_pipeline,
        _sample_latent,
    )

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    LATENT_DIR.mkdir(parents=True, exist_ok=True)
    EXAMPLE_DIR.mkdir(parents=True, exist_ok=True)
    seeds = sample_seeds()
    pending = [i for i in range(NUM_SAMPLES) if not sample_is_complete(i)]
    print(
        f"{NUM_SAMPLES - len(pending)} of {NUM_SAMPLES} samples already saved. "
        f"{len(pending)} samples still need generation and inversion.",
        flush=True,
    )
    print(
        f"Each remaining sample is 1 generation + {len(CONDITIONS)} inversions "
        f"({len(pending)} generations and {len(pending) * len(CONDITIONS)} inversions).",
        flush=True,
    )
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}", flush=True)

    dtype = _dtype(DTYPE)
    if pending:
        load_started = time.perf_counter()
        print("Loading the generation pipeline...", flush=True)
        pipe = _load_generation_pipeline(MODEL_ID, dtype, DEVICE)
        print(f"Generation pipeline ready in {fmt_duration(time.perf_counter() - load_started)}.", flush=True)
        if float(pipe.scheduler.init_noise_sigma) != 1.0:
            raise RuntimeError("DDIM init_noise_sigma is not 1.")
        if pipe.vae_scale_factor != VAE_SCALE_FACTOR:
            raise RuntimeError(f"VAE scale factor {pipe.vae_scale_factor} != {VAE_SCALE_FACTOR}.")
        gen_clock = StepClock(len(pending), "generation")
        generated = []
        try:
            for sample_id in pending:
                gen_clock.begin()
                z = _sample_latent(pipe, seeds[sample_id], IMAGE_SIZE, IMAGE_SIZE, dtype)
                result = pipe(
                    prompt=PROMPT,
                    latents=z.to(DEVICE),
                    num_inference_steps=NUM_INFERENCE_STEPS,
                    guidance_scale=GUIDANCE_SCALE,
                    height=IMAGE_SIZE,
                    width=IMAGE_SIZE,
                )
                if len(result.images) != 1:
                    raise RuntimeError(f"sample {sample_id} returned {len(result.images)} images")
                image = result.images[0].convert("RGB")
                if image.size != (IMAGE_SIZE, IMAGE_SIZE):
                    raise RuntimeError(f"sample {sample_id} image size {image.size}")
                generated.append((sample_id, z.detach().to(device="cpu", dtype=torch.float32), image))
                gen_clock.end(f"sample {sample_id + 1}/{NUM_SAMPLES}, seed {seeds[sample_id]}")
        finally:
            _release_pipeline(pipe)
            print("Generation pipeline released.", flush=True)

        load_started = time.perf_counter()
        print("Loading the inversion pipeline...", flush=True)
        inverse = _load_inversion_pipeline(MODEL_ID, dtype, DEVICE)
        print(f"Inversion pipeline ready in {fmt_duration(time.perf_counter() - load_started)}.", flush=True)
        if type(inverse.inverse_scheduler).__name__ != "DDIMInverseScheduler":
            raise RuntimeError(f"unexpected inverse scheduler {type(inverse.inverse_scheduler).__name__}")
        inv_clock = StepClock(len(generated) * len(CONDITIONS), "inversion")
        try:
            for sample_id, z, image in generated:
                blob = {"z": z}
                sample_started = time.perf_counter()
                for cond_index, (kind, magnitude) in enumerate(CONDITIONS):
                    inv_clock.begin()
                    torch.manual_seed(SEED + 1000 * sample_id + cond_index)
                    if torch.cuda.is_available():
                        torch.cuda.manual_seed_all(SEED + 1000 * sample_id + cond_index)
                    transformed = geom.apply_geometric(image, kind, magnitude)
                    if sample_id == 0:
                        label = "baseline" if kind == "baseline" else f"{kind}_{magnitude:g}"
                        transformed.save(EXAMPLE_DIR / f"sample000_{label}.png")
                    inverted = inverse.invert(
                        prompt=PROMPT,
                        image=transformed,
                        num_inference_steps=NUM_INFERENCE_STEPS,
                        guidance_scale=GUIDANCE_SCALE,
                        inpaint_strength=1.0,
                        num_reg_steps=0,
                    )
                    z_hat = _recovered_latent(inverted.latents, z.shape)
                    z_hat = z_hat.detach().to(device="cpu", dtype=torch.float32).contiguous()
                    if tuple(z_hat.shape) != tuple(z.shape) or not torch.isfinite(z_hat).all():
                        raise RuntimeError(f"bad inverted latent for sample {sample_id} {kind} {magnitude}")
                    blob[latent_key(kind, magnitude)] = z_hat
                    received = geom.magnitude_correlation(z.numpy(), z_hat.numpy())
                    undone = undo_image_transform(z_hat.numpy(), kind, magnitude, VAE_SCALE_FACTOR)
                    corrected = geom.magnitude_correlation(z.numpy(), undone)
                    inv_clock.end(
                        f"sample {sample_id + 1}/{NUM_SAMPLES}, {kind} {magnitude:g} | "
                        f"received {received:.3f}, undone {corrected:.3f}"
                    )
                torch.save(blob, sample_path(sample_id))
                print(
                    f"Saved sample {sample_id + 1}/{NUM_SAMPLES} "
                    f"({fmt_duration(time.perf_counter() - sample_started)} for its {len(CONDITIONS)} inversions).",
                    flush=True,
                )
                _write_table_and_metadata(_git_commit())
        finally:
            _release_pipeline(inverse)
            print("Inversion pipeline released.", flush=True)
    else:
        _write_table_and_metadata(None)
    frame = pd.read_csv(OUTPUT_DIR / "correlations.csv")
    print(f"Rows in {OUTPUT_DIR / 'correlations.csv'}: {len(frame)}", flush=True)
    return frame


def _write_table_and_metadata(git_commit: str | None) -> None:
    rows = []
    complete = [i for i in range(NUM_SAMPLES) if sample_is_complete(i)]
    for sample_id in complete:
        for row in _scores_for_blob(_load_saved(sample_id)):
            row["sample_id"] = sample_id
            rows.append(row)
    frame = pd.DataFrame(rows)
    csv_path = OUTPUT_DIR / "correlations.csv"
    frame.to_csv(csv_path, index=False)
    meta_path = OUTPUT_DIR / "metadata.json"
    if git_commit is None and meta_path.is_file():
        git_commit = json.loads(meta_path.read_text(encoding="utf-8")).get("git_commit")
    metadata = {
        "seed": SEED,
        "num_samples": NUM_SAMPLES,
        "samples_complete": complete,
        "sample_seeds": sample_seeds(),
        "reference_run": "results/geometric_registration/run_001 samples 0-4",
        "model_id": MODEL_ID,
        "image_size": IMAGE_SIZE,
        "vae_scale_factor": VAE_SCALE_FACTOR,
        "num_inference_steps": NUM_INFERENCE_STEPS,
        "guidance_scale": GUIDANCE_SCALE,
        "prompt": PROMPT,
        "dtype": DTYPE,
        "correlation_received": "pearsonr(|z|, |z_hat|)",
        "correlation_undone": "pearsonr(|z|, |undo(z_hat)|)",
        "undo": "opposite latent shift of n/8 pixels, or opposite latent rotation; fill 0",
        "conditions": [{"transformation_type": k, "magnitude": m} for k, m in CONDITIONS],
        "git_commit": git_commit,
    }
    (OUTPUT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")


run_experiment()


## 3. Correlation before and after the latent undo

Both lines use the saved tensors. Magnitude 0 is the untransformed image, where the undo does nothing. The band is the standard error across the five images. A higher undone line means the opposite latent shift or rotation moved `|z_hat|` closer to `|z|` under Pearson correlation.


In [ ]:
def load_scores() -> pd.DataFrame:
    missing = [i for i in range(NUM_SAMPLES) if not sample_is_complete(i)]
    if missing:
        raise FileNotFoundError(
            f"Samples {missing} are not saved in {LATENT_DIR}. Run the experiment cell on HiPerGator first."
        )
    rows = []
    for sample_id in range(NUM_SAMPLES):
        for row in _scores_for_blob(_load_saved(sample_id)):
            row["sample_id"] = sample_id
            rows.append(row)
    frame = pd.DataFrame(rows)
    csv_path = OUTPUT_DIR / "correlations.csv"
    if csv_path.is_file():
        saved = pd.read_csv(csv_path)
        merged = frame.merge(
            saved,
            on=["sample_id", "transformation_type", "magnitude"],
            suffixes=("_tensor", "_csv"),
        )
        if len(merged) != len(frame):
            raise RuntimeError("correlations.csv does not match the saved latents")
        drift = np.max(np.abs(merged.correlation_received_tensor - merged.correlation_received_csv))
        if drift > 1e-5:
            raise RuntimeError(f"CSV correlations differ from the tensors by {drift}")
    return frame


def _series(frame: pd.DataFrame, kind: str) -> pd.DataFrame:
    base = frame.loc[frame.transformation_type == "baseline"].copy()
    base["transformation_type"] = kind
    part = frame.loc[frame.transformation_type == kind]
    return pd.concat([base, part], ignore_index=True)


scores = load_scores()
print(f"loaded {scores.sample_id.nunique()} samples, {len(scores)} rows")

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, kind in zip(axes, FAMILIES):
    block = _series(scores, kind)
    for column, label in (
        ("correlation_received", "as inverted"),
        ("correlation_undone", "after latent undo"),
    ):
        grouped = block.groupby("magnitude")[column]
        mean = grouped.mean().sort_index()
        sem = grouped.std(ddof=1).sort_index() / np.sqrt(grouped.count().sort_index())
        ax.plot(mean.index, mean.to_numpy(), marker="o", label=label)
        ax.fill_between(mean.index, mean - sem, mean + sem, alpha=0.18)
    ax.set(xlabel=FAMILY_TITLES[kind], xticks=sorted(block.magnitude.unique()), ylim=(-0.15, 1.0))
    ax.legend(fontsize=8)
axes[0].set_ylabel("Pearson correlation of absolute latent values")
fig.suptitle("Five images. Magnitude 0 is the untransformed inversion.")
fig.tight_layout()
plt.show()

summary = (
    scores.groupby(["transformation_type", "magnitude"])[["correlation_received", "correlation_undone"]]
    .mean()
    .reset_index()
)
summary["gain"] = summary.correlation_undone - summary.correlation_received
summary.round(3)


## 4. Latent heatmaps for sample 0

Channels 0 and 1 of the original Gaussian `z`, the inverted latent `z_hat`, and the latent after the undo. Rows are the tested magnitudes, including no transform. Within one figure and one channel, every panel shares a diverging scale centered at 0. These are latent maps, not decoded images.


In [ ]:
blob0 = _load_saved(0)
z0 = blob0["z"].numpy()[0]


def _family_block(kind: str) -> list[tuple[str, float, np.ndarray, np.ndarray]]:
    rows = []
    mags = [0.0] if kind == "baseline" else [0.0] + [m for k, m in CONDITIONS if k == kind]
    for magnitude in mags:
        source_kind = "baseline" if magnitude == 0 else kind
        z_hat = blob0[latent_key(source_kind, magnitude)].numpy()[0]
        undone = undo_image_transform(z_hat[None], source_kind, magnitude, VAE_SCALE_FACTOR)[0]
        rows.append((source_kind, magnitude, z_hat, undone))
    return rows


for kind in FAMILIES:
    rows = _family_block(kind)
    limit = {0: 0.0, 1: 0.0}
    for channel in (0, 1):
        stacks = [z0[channel]]
        for _, _, z_hat, undone in rows:
            stacks.extend([z_hat[channel], undone[channel]])
        limit[channel] = float(max(np.max(np.abs(v)) for v in stacks))
        limit[channel] = max(limit[channel], 1e-6)
    fig, axes = plt.subplots(len(rows), 5, figsize=(12, 2.3 * len(rows)))
    for row_index, (source_kind, magnitude, z_hat, undone) in enumerate(rows):
        column_specs = [
            (0, z0, "z ch 0"),
            (0, z_hat, "z_hat ch 0"),
            (0, undone, "undone ch 0"),
            (1, z_hat, "z_hat ch 1"),
            (1, undone, "undone ch 1"),
        ]
        for col, (channel, latent, title) in enumerate(column_specs):
            axes[row_index, col].imshow(
                latent[channel], cmap="coolwarm", vmin=-limit[channel], vmax=limit[channel]
            )
            axes[row_index, col].set_title(f"{title}\n{source_kind} {magnitude:g}", fontsize=8)
            axes[row_index, col].axis("off")
    fig.colorbar(axes[0, 2].images[0], ax=axes[:, 0:3].ravel().tolist(), fraction=0.02)
    fig.colorbar(axes[0, 4].images[0], ax=axes[:, 3:5].ravel().tolist(), fraction=0.03)
    fig.suptitle(f"Sample 0, {FAMILY_TITLES[kind]}. Shared scale within each channel.")
    fig.tight_layout()
    plt.show()


## 5. Reading the five-image table

`gain` is the undone correlation minus the received correlation. Positive gain means the latent-space undo increased agreement with `|z|`. With five images, a gain of a few hundredths can be noise; the standard error on the graph is the check.


In [ ]:
print(f"Baseline mean received {scores.loc[scores.transformation_type == 'baseline', 'correlation_received'].mean():.3f}")
for kind in FAMILIES:
    block = summary.loc[summary.transformation_type == kind].sort_values("magnitude")
    for _, rec in block.iterrows():
        print(
            f"{kind} {rec.magnitude:g}: received {rec.correlation_received:.3f}, "
            f"undone {rec.correlation_undone:.3f}, gain {rec.gain:.3f}"
        )
n_positive = int((summary.loc[summary.transformation_type != "baseline", "gain"] > 0).sum())
n_rows = int((summary.transformation_type != "baseline").sum())
print(f"Undo increased the mean correlation in {n_positive} of {n_rows} transformed conditions.")
